In [ ]:
!pip install flask huggingface-hub pypdf python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pypdf]


In [ ]:
!pip install llama-cpp-python \
--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
###for cpu

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 33.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


In [ ]:
#for gpu
!pip install "https://huggingface.co/vinamrag-code/llama-cpp-python-cuda-wheels/resolve/main/llama_cpp_python-0.3.36%2Bcu130.gemma2-py3-none-linux_x86_64.whl"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.3/271.3 MB 38.2 MB/s  0:00:07
  Attempting uninstall: llama-cpp-python
    Found existing installation: llama_cpp_python 0.3.36
    Uninstalling llama_cpp_python-0.3.36:
      Successfully uninstalled llama_cpp_python-0.3.36


In [ ]:
import json
from flask import jsonify
from pypdf import PdfReader
from llama_cpp import Llama
from huggingface_hub import hf_hub_download

In [ ]:
MODEL_REPO = "Qwen/Qwen2.5-3B-Instruct-GGUF"
MODEL_FILE = "qwen2.5-3b-instruct-q4_k_m.gguf"
model_path = hf_hub_download(
                repo_id=MODEL_REPO,
                filename=MODEL_FILE
            )

model = Llama(
            model_path=model_path,
            n_ctx=8192,
            n_batch=1024,
            n_threads=4,
            n_gpu_layers=-1,
            verbose=False
        )

In [ ]:
def upload(path):
        reader = PdfReader(path)
        extracted_txt = ""
        all_rules = []
        i=1
        for p in reader.pages:
          print("Page:",i)
          page_txt = p.extract_text(extraction_mode="layout")
          if not page_txt or not page_txt.strip():
            print("Empty page")
            i += 1
            continue

          prompt = f"""
          You are a labor law analysis and information extraction system.

          Analyze the following page from a labor law document.

          Identify all explicitly stated legal rules, obligations, rights,
          restrictions, prohibitions, and conditions relevant to employment
          relationships.

          Extract each distinct legal rule separately.

          For each rule, extract:

          1. rule_name
          2. description
          3. condition
          4. operator
          5. value
          6. unit
          7. applicability
          8. exceptions
          9. legal_reference
          10. original_text
          11. rule_type
          12. contract_must_state
          13. source_page

          RULE CLASSIFICATION:

          Classify each rule using exactly one of the following types:

          - contractual_requirement:
            A requirement concerning information, terms, or conditions
            that must be explicitly included in an employment contract.

          - minimum_employee_right:
            A statutory minimum right or protection granted to an employee.
            The contract must not contradict or reduce this right.

          - employer_obligation:
            A legal obligation imposed on the employer that does not
            necessarily need to be explicitly stated in the contract.

          - employee_obligation:
            A legal obligation or restriction imposed on the employee.

          - special_category:
            A rule applicable only to specific employees, occupations,
            industries, or employment circumstances.

          - definition:
            A legal definition or scope provision that establishes
            the meaning or applicability of other legal provisions.

          CONTRACT CHECKING:
          CONTRACT REVIEWABILITY:

The field contract_reviewable must be:

- true: When the legal rule can be evaluated by examining an
  individual employment contract, including whether its terms
  satisfy, contradict, or reduce a statutory requirement or right.

- false: When the rule concerns general administrative procedures,
  recruitment processes, workplace inspections, establishment-level
  obligations, government procedures, or matters that cannot
  reasonably be evaluated from the employment contract alone.

IMPORTANT:
- A rule can be contract_reviewable even when contract_must_state is false.
- minimum_employee_right rules can be contract_reviewable.
- employer_obligation rules can be contract_reviewable if their
  requirements can be checked against contract terms.
- Do not mark a rule true merely because it applies to employers
  or employment relationships.
- Evaluate each rule independently.

          The field contract_must_state must be:

          - true: When the law explicitly requires a term, condition,
            or piece of information to be included in the employment contract.

          - false: When the rule establishes a legal right, employer
            obligation, employee obligation, restriction, or definition
            that does not explicitly require its inclusion in the contract.

          Do not classify a rule as contractual_requirement merely
          because it is relevant to employment contracts.

          EXTRACTION RULES:

          - Extract only explicitly stated legal rules.
          - Do not invent or assume legal conditions.
          - Preserve all exceptions, qualifications, and limitations.
          - Include rules even if they do not contain numerical values.
          - Extract multiple distinct rules from one page separately.
          - Do not combine different legal obligations into one rule.
          - Do not extract general explanations that establish no
            legal obligation, right, restriction, or condition.
          - Preserve the original legal meaning.
          - Do not remove important conditions from the original text.
          - Do not duplicate the same legal rule.
          - A rule appearing on multiple pages should retain its
            original legal reference and the page where its text appears.

          STRUCTURED VALUES:

          - Extract numerical limits as numbers, not words.
          - Use operator values such as:
            ">", ">=", "<", "<=", "=", "between", or null.
          - If a rule has no numerical condition, use null for
            condition, operator, value, and unit where appropriate.
          - Use null for missing or unspecified values.
          - Do not use "N/A", empty strings, or invented values.

          LEGAL REFERENCES:

          - Extract the article number whenever explicitly available.
          - Use a consistent format such as "Article 53".
          - If the article number is unavailable on the page, use null.
          - Do not use the entire legal provision as the legal_reference.

          SOURCE PAGE:

          - source_page must contain the actual page number provided
            by the document-processing system.
          - If the page number is unavailable, use null.

          OUTPUT REQUIREMENTS:

          Return valid JSON only, without Markdown or explanations.

          Use this exact structure:

          {{
              "rules": [
                  {{
                      "rule_name": "",
                      "description": "",
                      "condition": null,
                      "operator": null,
                      "value": null,
                      "unit": null,
                      "applicability": "",
                      "exceptions": null,
                      "legal_reference": null,
                      "original_text": "",
                      "rule_type": "",
                      "contract_must_state": false,
                      "contract_reviewable": false,
                      "source_page": null
                  }}
              ]
          }}

          If no relevant legal rules exist, return:

          {{
              "rules": []
          }}

          Page text:
          {page_txt}
          """


          response = model.create_chat_completion(
          messages=[
              {
                  "role": "user",
                  "content": prompt
              }
          ],
          temperature=0.0,
          max_tokens=2000,
          response_format={
              "type": "json_object"
              }
          )

          result = response["choices"][0]["message"]["content"]

          try:
              result = json.loads(result)

          except json.JSONDecodeError as e:
              result = {"rules": []}

          if result and result.get("rules"):
                    all_rules.extend(result["rules"])
          i+=1

        return all_rules








In [ ]:
rules = upload("labor_law.pdf")

Page: 1
Page: 2
Page: 3
Page: 4
Page: 5
Page: 6
Page: 7
Page: 8
Page: 9
Page: 10
Page: 11
Page: 12
Page: 13
Page: 14
Page: 15
Page: 16
Page: 17
Page: 18
Page: 19
Page: 20
Page: 21
Page: 22
Page: 23
Page: 24
Page: 25
Page: 26
Page: 27
Page: 28
Page: 29
Page: 30
Page: 31
Page: 32
Page: 33
Page: 34
Page: 35
Page: 36
Page: 37
Page: 38
Page: 39
Page: 40
Page: 41
Page: 42
Page: 43
Page: 44
Page: 45
Page: 46
Page: 47


In [ ]:
with open("labor_rules.json", "w", encoding="utf-8") as f:
    json.dump(rules, f, ensure_ascii=False, indent=4)

In [ ]:
from pypdf import PdfReader
from docx import Document

def extract_text(path):
        prefix = path.split(".")[-1]
        if prefix == "pdf" or prefix == "PDF":
            reader = PdfReader(path)
            extracted_txt = ""
            for p in reader.pages:
                tmp = p.extract_text(extraction_mode="layout")

                if tmp:
                    extracted_txt += tmp + "\n"
            return extracted_txt
        doc = Document(path)
        extracted_txt = ""
        for paragraph in doc.paragraphs:
            if paragraph.text:
                extracted_txt += paragraph.text + "\n"
        return extracted_txt

In [ ]:
from llama_cpp import Llama
from huggingface_hub import hf_hub_download
import multiprocessing

MODEL_REPO = "Qwen/Qwen2.5-3B-Instruct-GGUF"
MODEL_FILE = "qwen2.5-3b-instruct-q4_k_m.gguf"
model_path = hf_hub_download(
         repo_id=MODEL_REPO,
        filename=MODEL_FILE
        )

model = Llama(
    model_path=model_path,
    n_ctx=8192,
    n_batch=512,
    n_threads=multiprocessing.cpu_count(),
    n_threads_batch=multiprocessing.cpu_count(),
    n_gpu_layers=0,
    verbose=False
)

In [ ]:
r = upload2("yes.docx")


In [ ]:
import json

with open("labor_rules.json", "r", encoding="utf-8") as file:
    rules = json.load(file)
contract_rules = [
    rule for rule in rules
    if rule.get("contract_reviewable") is True
]